# Qwen3-0.6B: IMDb LoRA Fine-Tuning

This notebook performs **LoRA fine-tuning** on `Qwen/Qwen3-0.6B` for binary IMDb sentiment classification:

* `negative`: label 0
* `positive`: label 1
* The original IMDb training set is split into training and validation sets using an **80% / 20%** ratio.
* The official IMDb test set remains separate and is used only for final evaluation.
* During training, the loss is calculated only on the assistant’s `positive` / `negative` responses.
* The final output is a compact **LoRA adapter**, and the notebook also demonstrates how to reload it.

Default full-training configuration: 20,000 training samples, 5,000 validation samples, 1 epoch, a maximum sequence length of 512, and a LoRA rank of 8.

After training is complete and the parameters have been saved, run the notebook through the first code block in Part 5. Then skip to Part 11 and continue running from there.


## 0. Install Dependencies

In [ ]:
# %pip uninstall -y torchao
# %pip install -q -U \
#   "transformers>=4.51.0,<6.0.0" \
#   "peft>=0.15.0,<1.0.0" \
#   "datasets>=3.0.0,<5.0.0" \
#   "accelerate>=1.0.0,<2.0.0" \
#   scikit-learn tqdm safetensors
# %pip install -q -U pandas

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00


> 如果安装后出现旧版本仍被占用的 import 错误，请执行 `Runtime → Restart session`，然后从下一格继续。


In [1]:
import gc
import inspect
import json
import os
import random
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from datasets import ClassLabel, Dataset, DatasetDict, load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
MODEL_NAME = "Qwen/Qwen3-0.6B"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("没有检测到 GPU。请在 Colab 中选择 Runtime → Change runtime type → T4 GPU。")


Python: 3.11.2
PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
GPU memory (GB): 8.0


## 1. Experiment Configuration


In [2]:
# ===== 最重要的开关 =====
QUICK_RUN = False

# 是否把最终 adapter 和结果保存到 Google Drive。
SAVE_TO_GOOGLE_DRIVE = True

# 训练超参数
MAX_SEQ_LENGTH = 512
MAX_REVIEW_TOKENS = 430
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8

# LoRA 超参数
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# 生成式最终测试的 batch size；显存不足时改为 8 或 4。
GENERATION_BATCH_SIZE = 16

# 断点续训：已有 checkpoint 且希望继续时设为 True。
RESUME_TRAINING = False

# 训练完后释放显存并从硬盘重新加载 adapter，以验证保存成功。
RELOAD_SAVED_ADAPTER = True

# 是否额外保存合并后的完整模型。默认关闭，因为文件明显更大。
MERGE_AND_SAVE_FULL_MODEL = False

if QUICK_RUN:
    TRAIN_LIMIT = 2_000
    VAL_LIMIT = 500
    FINAL_TEST_SAMPLES = 500
else:
    TRAIN_LIMIT = None          # 完整 20,000 条
    VAL_LIMIT = None            # 完整 5,000 条
    FINAL_TEST_SAMPLES = 25_000 # IMDb 官方完整测试集

print({
    "QUICK_RUN": QUICK_RUN,
    "TRAIN_LIMIT": TRAIN_LIMIT,
    "VAL_LIMIT": VAL_LIMIT,
    "FINAL_TEST_SAMPLES": FINAL_TEST_SAMPLES,
    "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
    "NUM_EPOCHS": NUM_EPOCHS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
})


{'QUICK_RUN': False, 'TRAIN_LIMIT': None, 'VAL_LIMIT': None, 'FINAL_TEST_SAMPLES': 25000, 'MAX_SEQ_LENGTH': 512, 'NUM_EPOCHS': 1, 'effective_batch_size': 32}


## 2. Set the Save Directory



In [3]:
IN_COLAB = "google.colab" in sys.modules

if SAVE_TO_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/qwen3_imdb_lora")
else:
    PROJECT_DIR = Path("/content/qwen3_imdb_lora" if IN_COLAB else "./qwen3_imdb_lora")

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
ADAPTER_DIR = PROJECT_DIR / "best_adapter"
RESULTS_DIR = PROJECT_DIR / "results"
MERGED_MODEL_DIR = PROJECT_DIR / "merged_full_model"

for folder in [CHECKPOINT_DIR, ADAPTER_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Adapter will be saved to:", ADAPTER_DIR)

# ============================================================
# Qwen3 基础模型目录
# ============================================================

if IN_COLAB:
    # Colab 保持原来的在线加载方式
    MODEL_SOURCE = MODEL_NAME

else:
    # 本地把基础模型放到：
    # 项目目录/base_model/Qwen3-0.6B
    from huggingface_hub import snapshot_download

    BASE_MODEL_DIR = (
        Path("./base_model") / "Qwen3-0.6B"
    )

    BASE_MODEL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("检查或下载基础模型：", MODEL_NAME)
    print("基础模型目录：", BASE_MODEL_DIR.resolve())

    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(BASE_MODEL_DIR),
    )

    MODEL_SOURCE = str(BASE_MODEL_DIR.resolve())
    
print("Model source:", MODEL_SOURCE)


Project directory: qwen3_imdb_lora
Adapter will be saved to: qwen3_imdb_lora\best_adapter
检查或下载基础模型： Qwen/Qwen3-0.6B
基础模型目录： E:\StudyFile\2026-T2\COMP9444-Neural Networks, Deep Learning\Group Project\Qwen3_0.6B_IMDB\base_model\Qwen3-0.6B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model source: E:\StudyFile\2026-T2\COMP9444-Neural Networks, Deep Learning\Group Project\Qwen3_0.6B_IMDB\base_model\Qwen3-0.6B


## 3. Load the Tokenizer and IMDb Dataset

The notebook first attempts to use `Kwaai/IMDB_Sentiment`, as in the original notebook. If it is unavailable, it automatically falls back to the standard `stanfordnlp/imdb` dataset.



In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# 训练时右侧 padding；生成时会临时改成左侧 padding。
tokenizer.padding_side = "right"

try:
    raw_dataset = load_dataset("Kwaai/IMDB_Sentiment", "plain_text")
    DATASET_USED = "Kwaai/IMDB_Sentiment"
except Exception as exc:
    print("项目镜像加载失败，改用 stanfordnlp/imdb。")
    print("原始错误：", repr(exc))
    raw_dataset = load_dataset("stanfordnlp/imdb")
    DATASET_USED = "stanfordnlp/imdb"

print("Dataset:", DATASET_USED)
print(raw_dataset)
print(raw_dataset["train"][0])


README.md:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

c:\Users\Rain\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rain\.cache\huggingface\hub\datasets--Kwaai--IMDB_Sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset: Kwaai/IMDB_Sentiment
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In 

## 4. Stratified 80% / 20% Train–Validation Split

The official IMDb test set is not used for training or hyperparameter tuning.


In [5]:
source_train = raw_dataset["train"]

# stratify_by_column 要求标签为 ClassLabel；必要时自动转换。
if not isinstance(source_train.features["label"], ClassLabel):
    source_train = source_train.class_encode_column("label")

split = source_train.train_test_split(
    test_size=0.20,
    seed=SEED,
    stratify_by_column="label",
)

train_raw = split["train"].shuffle(seed=SEED)
val_raw = split["test"].shuffle(seed=SEED)
test_raw = raw_dataset["test"].shuffle(seed=SEED)

if TRAIN_LIMIT is not None:
    train_raw = train_raw.select(range(min(TRAIN_LIMIT, len(train_raw))))
if VAL_LIMIT is not None:
    val_raw = val_raw.select(range(min(VAL_LIMIT, len(val_raw))))
if FINAL_TEST_SAMPLES is not None:
    test_raw = test_raw.select(range(min(FINAL_TEST_SAMPLES, len(test_raw))))

splits_df = pd.DataFrame([
    {
        "split": name,
        "samples": len(ds),
        "negative": sum(int(x) == 0 for x in ds["label"]),
        "positive": sum(int(x) == 1 for x in ds["label"]),
    }
    for name, ds in [("train", train_raw), ("validation", val_raw), ("official_test", test_raw)]
])
display(splits_df)


,split,samples,negative,positive
0,train,20000,10000,10000
1,validation,5000,2500,2500
2,official_test,25000,12500,12500


## 5. Prepare the Supervised Fine-Tuning Data

The training format follows the same prompt structure as the original notebook:

```text
system: Classify the overall sentiment ...
user: <IMDb review>
assistant: positive / negative
```

All prompt tokens in the labels are set to `-100`, so the loss is calculated only on the assistant’s response.


In [6]:
SYSTEM_PROMPT = (
    "Classify the overall sentiment of the IMDb movie review. "
    "Reply with exactly one word: positive or negative."
)

ID_TO_LABEL = {0: "negative", 1: "positive"}
LABEL_TO_ID = {"negative": 0, "positive": 1}


def clean_review(text: str) -> str:
    text = re.sub(r"<br\s*/?>", " ", str(text), flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def truncate_review(text: str, max_tokens: int = MAX_REVIEW_TOKENS) -> str:
    """保留影评开头与结尾，避免只保留开头造成信息损失。"""
    text = clean_review(text)
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    if len(token_ids) <= max_tokens:
        return text

    head_len = max_tokens // 2
    tail_len = max_tokens - head_len
    head = tokenizer.decode(token_ids[:head_len], skip_special_tokens=True)
    tail = tokenizer.decode(token_ids[-tail_len:], skip_special_tokens=True)
    return head + "\n\n[... middle content omitted ...]\n\n" + tail


def build_prompt_messages(review: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": review},
    ]


def common_prefix_length(a: list[int], b: list[int]) -> int:
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def encode_training_example(text: str, label: int) -> dict[str, list[int]]:
    # 若模板开销比预期大，自动进一步缩短 review，确保答案仍在序列内。
    review_budget = MAX_REVIEW_TOKENS

    for _ in range(5):
        review = truncate_review(text, max_tokens=max(32, review_budget))
        prompt_messages = build_prompt_messages(review)
        full_messages = prompt_messages + [
            {"role": "assistant", "content": ID_TO_LABEL[int(label)]}
        ]

        # Transformers v5 的 apply_chat_template 默认返回 BatchEncoding；
        # 显式提取 input_ids，确保写入 Dataset 的都是普通 Python int 列表。
        prompt_encoded = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
        )
        full_encoded = tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=False,
            enable_thinking=False,
            return_dict=True,
        )

        prompt_ids = prompt_encoded["input_ids"]
        full_ids = full_encoded["input_ids"]

        # 兼容 tensor / ndarray / 单样本二维列表等返回形式。
        if hasattr(prompt_ids, "tolist"):
            prompt_ids = prompt_ids.tolist()
        if hasattr(full_ids, "tolist"):
            full_ids = full_ids.tolist()

        if prompt_ids and isinstance(prompt_ids[0], (list, tuple)):
            prompt_ids = prompt_ids[0]
        if full_ids and isinstance(full_ids[0], (list, tuple)):
            full_ids = full_ids[0]

        prompt_ids = [int(token_id) for token_id in prompt_ids]
        full_ids = [int(token_id) for token_id in full_ids]

        if len(full_ids) <= MAX_SEQ_LENGTH:
            break

        review_budget -= len(full_ids) - MAX_SEQ_LENGTH + 8
    else:
        raise RuntimeError("无法把样本压缩到 MAX_SEQ_LENGTH；请调大长度或调小 review token 数。")

    # 正常情况下 prompt_ids 是 full_ids 的严格前缀；使用公共前缀提高模板兼容性。
    answer_start = common_prefix_length(prompt_ids, full_ids)
    labels = [-100] * answer_start + full_ids[answer_start:]

    if not any(x != -100 for x in labels):
        raise RuntimeError("该样本没有可训练的 assistant token。")

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "length": len(full_ids),
    }


def tokenize_batch(batch: dict[str, list[Any]]) -> dict[str, list[list[int]]]:
    encoded = [
        encode_training_example(text, int(label))
        for text, label in zip(batch["text"], batch["label"])
    ]
    return {key: [row[key] for row in encoded] for key in encoded[0]}


In [8]:
print("正在 tokenization；完整 25,000 条通常需要几分钟……")

train_tokenized = train_raw.map(
    tokenize_batch,
    batched=True,
    batch_size=128,
    writer_batch_size=128,
    remove_columns=train_raw.column_names,
    desc="Tokenizing train",
)

val_tokenized = val_raw.map(
    tokenize_batch,
    batched=True,
    batch_size=128,
    writer_batch_size=128,
    remove_columns=val_raw.column_names,
    desc="Tokenizing validation",
)

print(train_tokenized)
print(val_tokenized)
print("Train length percentiles:", np.percentile(train_tokenized["length"], [50, 90, 95, 99, 100]))

sample = train_tokenized[0]
answer_tokens = [token for token, label in zip(sample["input_ids"], sample["labels"]) if label != -100]
print("Supervised answer tokens:", tokenizer.decode(answer_tokens, skip_special_tokens=False))


正在 tokenization；完整 25,000 条通常需要几分钟……


Tokenizing train:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'length'],
    num_rows: 20000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'length'],
    num_rows: 5000
})
Train length percentiles: [256. 479. 479. 479. 479.]
Supervised answer tokens: negative<|im_end|>



## 6. Data Collator with Dynamic Padding

Each batch is padded only to the length of the longest sample in that batch, reducing unnecessary computation.


In [9]:
@dataclass
class CausalLMCollator:
    pad_token_id: int
    label_pad_token_id: int = -100

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        max_len = max(len(feature["input_ids"]) for feature in features)
        batch_size = len(features)

        input_ids = torch.full(
            (batch_size, max_len),
            self.pad_token_id,
            dtype=torch.long,
        )
        attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
        labels = torch.full(
            (batch_size, max_len),
            self.label_pad_token_id,
            dtype=torch.long,
        )

        for i, feature in enumerate(features):
            length = len(feature["input_ids"])
            input_ids[i, :length] = torch.tensor(feature["input_ids"], dtype=torch.long)
            attention_mask[i, :length] = torch.tensor(feature["attention_mask"], dtype=torch.long)
            labels[i, :length] = torch.tensor(feature["labels"], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


data_collator = CausalLMCollator(pad_token_id=tokenizer.pad_token_id)

# 快速检查一个 batch 的形状与监督 token 数量。
check_batch = data_collator([train_tokenized[i] for i in range(min(3, len(train_tokenized)))])
print({key: tuple(value.shape) for key, value in check_batch.items()})
print("Supervised tokens per example:", (check_batch["labels"] != -100).sum(dim=1).tolist())


{'input_ids': (3, 479), 'attention_mask': (3, 479), 'labels': (3, 479)}
Supervised tokens per example: [3, 3, 3]


## 7. Load the Base Model and Add a LoRA Adapter

The base model parameters are frozen, and only the newly added LoRA parameters are trained. Setting `target_modules="all-linear"` applies LoRA to the linear layers in the Transformer without replacing the output layer.


In [ ]:
# T4 使用 float16；支持 bf16 的新 GPU 会自动使用 bfloat16。
BF16_SUPPORTED = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
MODEL_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_SOURCE,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

base_model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    bias="none",
)

model = get_peft_model(base_model, lora_config)

# 冻结大部分参数时，gradient checkpointing 可能需要输入 embedding 的梯度入口。
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

model.print_trainable_parameters()
print("Model dtype:", MODEL_DTYPE)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395
Model dtype: torch.bfloat16


## 8. Configure the Trainer

To maintain compatibility across different versions of Transformers, the code automatically selects either the `eval_strategy` parameter or its legacy name, `evaluation_strategy`.


In [13]:
training_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_strategy="steps",
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not BF16_SUPPORTED,
    bf16=BF16_SUPPORTED,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    length_column_name="length",
    seed=SEED,
    data_seed=SEED,
)

training_args_signature = inspect.signature(
    TrainingArguments.__init__
).parameters

if "overwrite_output_dir" in training_args_signature:
    training_kwargs["overwrite_output_dir"] = False

if "eval_strategy" in training_args_signature:
    training_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in training_args_signature:
    training_kwargs["evaluation_strategy"] = "epoch"

if "train_sampling_strategy" in training_args_signature:
    training_kwargs["train_sampling_strategy"] = "group_by_length"
elif "group_by_length" in training_args_signature:
    training_kwargs["group_by_length"] = True

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Training steps per epoch (approximately):", len(trainer.get_train_dataloader()) // GRADIENT_ACCUMULATION_STEPS)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training steps per epoch (approximately): 625


## 9. Start Training

The full configuration typically takes a considerable amount of time to run on a T4 GPU. During training, Colab continuously displays the loss, learning rate, and progress.


In [14]:
resume_arg = None

if RESUME_TRAINING:
    checkpoints = sorted(
        CHECKPOINT_DIR.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if checkpoints:
        resume_arg = str(checkpoints[-1])
        print("Resume from:", resume_arg)
    else:
        print("没有找到 checkpoint，将从头训练。")

train_start = time.perf_counter()
train_result = trainer.train(resume_from_checkpoint=resume_arg)
train_seconds = time.perf_counter() - train_start

print("Training finished.")
print("Elapsed minutes:", round(train_seconds / 60, 2))
print(train_result.metrics)


Epoch,Training Loss,Validation Loss
1,0.050200,0.049310


Training finished.
Elapsed minutes: 211.81
{'train_runtime': 12708.0028, 'train_samples_per_second': 1.574, 'train_steps_per_second': 0.049, 'total_flos': 1.5494810639007744e+16, 'train_loss': 0.05608323345184326, 'epoch': 1.0}


## 10. Validation Loss, Training Logs, and Saving the Best Adapter

With `load_best_model_at_end=True`, the checkpoint with the lowest validation loss is restored when training finishes. It is then saved to `best_adapter`.


In [15]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

# 保存 LoRA adapter，而不是复制一整份基础模型。
trainer.model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)

run_summary = {
    "model_name": MODEL_NAME,
    "dataset": DATASET_USED,
    "seed": SEED,
    "quick_run": QUICK_RUN,
    "train_samples": len(train_raw),
    "validation_samples": len(val_raw),
    "official_test_samples_selected": len(test_raw),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_review_tokens": MAX_REVIEW_TOKENS,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "training_seconds": train_seconds,
    "train_metrics": train_result.metrics,
    "eval_metrics": eval_metrics,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

with open(RESULTS_DIR / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(run_summary, f, ensure_ascii=False, indent=2, default=str)

log_df = pd.DataFrame(trainer.state.log_history)
log_df.to_csv(RESULTS_DIR / "trainer_log_history.csv", index=False)

display(log_df.tail(10))
print("Saved adapter files:")
for path in sorted(ADAPTER_DIR.iterdir()):
    print(" -", path.name, f"({path.stat().st_size / 1024**2:.2f} MB)")


Training Loss,Validation Loss,Epoch
0.050200,0.049310,1


{'eval_loss': 0.0493096187710762}


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
18,0.044303,0.435963,1.455314e-05,0.76,475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,0.046882,0.499036,1.029292e-05,0.80,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,0.042060,0.985941,6.698730e-06,0.84,525,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,0.047800,0.254622,3.830861e-06,0.88,550,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,0.057703,0.940590,1.737415e-06,0.92,575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,0.043848,0.369041,4.535066e-07,0.96,600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,0.050200,0.278087,6.718828e-10,1.00,625,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,NaN,NaN,NaN,1.00,625,0.04931,960.6160,5.205,0.651,NaN,NaN,NaN,NaN,NaN
26,NaN,NaN,NaN,1.00,625,NaN,NaN,NaN,NaN,12708.0028,1.574,0.049,1.549481e+16,0.056083
27,NaN,NaN,NaN,1.00,625,0.04931,959.7821,5.210,0.651,NaN,NaN,NaN,NaN,NaN


Saved adapter files:
 - README.md (0.00 MB)
 - adapter_config.json (0.00 MB)
 - adapter_model.safetensors (19.30 MB)
 - chat_template.jinja (0.00 MB)
 - tokenizer.json (10.89 MB)
 - tokenizer_config.json (0.00 MB)


## 11. Free GPU Memory and Reload the Saved Adapter

This cell verifies that the saved adapter files can be reloaded independently. The base model, `Qwen/Qwen3-0.6B`, is still required for future use.


In [7]:
# 在全新 Colab 运行时中重新确定模型精度
BF16_SUPPORTED = bool(
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

MODEL_DTYPE = (
    torch.bfloat16
    if BF16_SUPPORTED
    else torch.float16
)

if RELOAD_SAVED_ADAPTER:
    # 兼容两种情况：
    # 1. 刚训练完，变量仍在内存；
    # 2. Colab 已重启，这些变量已经不存在。
    for variable_name in [
        "trainer",
        "model",
        "base_model",
        "reloaded_base",
    ]:
        if variable_name in globals():
            del globals()[variable_name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    adapter_config_path = ADAPTER_DIR / "adapter_config.json"
    adapter_weights_path = ADAPTER_DIR / "adapter_model.safetensors"

    if not adapter_config_path.exists():
        raise FileNotFoundError(
            f"没有找到 adapter 配置：{adapter_config_path}"
        )

    if not adapter_weights_path.exists():
        raise FileNotFoundError(
            f"没有找到 adapter 权重：{adapter_weights_path}"
        )

    print("正在加载 Qwen3 基础模型……")

    reloaded_base = AutoModelForCausalLM.from_pretrained(
        MODEL_SOURCE,
        torch_dtype=MODEL_DTYPE,
        low_cpu_mem_usage=True,
    )

    print("正在加载训练好的 LoRA adapter……")

    model = PeftModel.from_pretrained(
        reloaded_base,
        ADAPTER_DIR,
        is_trainable=False,
    )

    model.to("cuda")
    model.eval()
    model.config.use_cache = True

    print("Adapter 加载成功：", ADAPTER_DIR)
    print("模型设备：", next(model.parameters()).device)
    print("模型精度：", MODEL_DTYPE)

else:
    # 这个分支只适用于训练仍在当前运行时内存中的情况
    if "trainer" not in globals():
        raise RuntimeError(
            "当前没有 trainer。重新连接后请把 "
            "RELOAD_SAVED_ADAPTER 设置为 True。"
        )

    model = trainer.model
    model.eval()
    model.config.use_cache = True

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


正在加载 Qwen3 基础模型……


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

正在加载训练好的 LoRA adapter……
Adapter 加载成功： qwen3_imdb_lora\best_adapter
模型设备： cuda:0
模型精度： torch.bfloat16


## 12. Generative Classification Function

As in the original zero-shot and few-shot notebooks, classification is performed by generating either `positive` or `negative`.


In [8]:
def parse_label(raw_output: str):
    match = re.search(r"\b(positive|negative)\b", raw_output.lower())
    return None if match is None else LABEL_TO_ID[match.group(1)]


def build_inference_prompt(review: str) -> str:
    review = truncate_review(review, max_tokens=MAX_REVIEW_TOKENS)
    return tokenizer.apply_chat_template(
        build_prompt_messages(review),
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


@torch.inference_mode()
def predict_batch(reviews: list[str]) -> list[dict[str, Any]]:
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    prompts = [build_inference_prompt(review) for review in reviews]
    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    torch.cuda.synchronize()
    start = time.perf_counter()

    generated = model.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    input_width = inputs["input_ids"].shape[1]
    output_texts = tokenizer.batch_decode(
        generated[:, input_width:],
        skip_special_tokens=True,
    )

    tokenizer.padding_side = old_padding_side

    return [
        {
            "raw_output": text.strip(),
            "prediction": parse_label(text),
            "seconds": elapsed / len(reviews),
            "prompt_tokens": int(inputs["attention_mask"][i].sum().item()),
        }
        for i, text in enumerate(output_texts)
    ]


# 单条 smoke test
sample = test_raw[0]
result = predict_batch([sample["text"]])[0]
print("True label:", ID_TO_LABEL[int(sample["label"])])
print("Prediction:", result)
print("Review preview:", clean_review(sample["text"])[:500])


True label: positive
Prediction: {'raw_output': 'positive', 'prediction': 1, 'seconds': 1.5391107000177726, 'prompt_tokens': 401}
Review preview: When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong? Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the cha


## 13. Evaluate on the Official Test Set

In full mode, all 25,000 test samples are evaluated by default; in quick mode, 500 samples are evaluated. Outputs that cannot be parsed are counted as incorrect when calculating the overall accuracy.


In [9]:
# adapter 已经保存，接下来只做推理，因此在内存中合并以提高速度
if isinstance(model, PeftModel):
    model = model.merge_and_unload(safe_merge=True)

model.eval()
model.config.use_cache = True

print("Inference model:", type(model).__name__)
print("Device:", next(model.parameters()).device)

def evaluate_generation(
    eval_dataset: Dataset,
    batch_size: int = GENERATION_BATCH_SIZE,
) -> tuple[pd.DataFrame, dict[str, float]]:
    rows = []

    for start_idx in tqdm(range(0, len(eval_dataset), batch_size), desc="Official test"):
        end_idx = min(start_idx + batch_size, len(eval_dataset))
        batch = eval_dataset[start_idx:end_idx]
        outputs = predict_batch(batch["text"])

        for offset, output in enumerate(outputs):
            true_label = int(batch["label"][offset])
            prediction = output["prediction"]
            rows.append({
                "index": start_idx + offset,
                "true_label": true_label,
                "true_label_text": ID_TO_LABEL[true_label],
                "prediction": prediction,
                "prediction_text": None if prediction is None else ID_TO_LABEL[prediction],
                "correct": prediction == true_label,
                "raw_output": output["raw_output"],
                "prompt_tokens": output["prompt_tokens"],
                "seconds": output["seconds"],
                "review_preview": clean_review(batch["text"][offset])[:1000],
            })

    frame = pd.DataFrame(rows)
    valid_mask = frame["prediction"].notna()
    valid_frame = frame.loc[valid_mask].copy()

    metrics = {
        "n_samples": int(len(frame)),
        "coverage": float(valid_mask.mean()),
        "overall_accuracy": float(frame["correct"].mean()),
        "mean_seconds_per_review": float(frame["seconds"].mean()),
        "mean_prompt_tokens": float(frame["prompt_tokens"].mean()),
    }

    if len(valid_frame):
        y_true = valid_frame["true_label"].astype(int)
        y_pred = valid_frame["prediction"].astype(int)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )
        metrics.update({
            "valid_only_accuracy": float(accuracy_score(y_true, y_pred)),
            "precision_positive": float(precision),
            "recall_positive": float(recall),
            "f1_positive": float(f1),
        })

    return frame, metrics


test_predictions, test_metrics = evaluate_generation(test_raw)
display(pd.DataFrame([test_metrics]))


Inference model: Qwen3ForCausalLM
Device: cuda:0


Official test:   0%|          | 0/1563 [00:00<?, ?it/s]

,n_samples,coverage,overall_accuracy,mean_seconds_per_review,mean_prompt_tokens,valid_only_accuracy,precision_positive,recall_positive,f1_positive
0,25000,1.0,0.95144,0.050903,282.72528,0.95144,0.957516,0.9448,0.951115


## 14. Confusion Matrix, Classification Report, and Misclassified Samples


In [10]:
valid_predictions = test_predictions.dropna(subset=["prediction"]).copy()
valid_predictions["prediction"] = valid_predictions["prediction"].astype(int)

print("Confusion matrix:")
print(confusion_matrix(
    valid_predictions["true_label"],
    valid_predictions["prediction"],
))
print()
print(classification_report(
    valid_predictions["true_label"],
    valid_predictions["prediction"],
    target_names=["negative", "positive"],
    digits=4,
    zero_division=0,
))

print("错误或无法解析的前 20 条：")
display(
    test_predictions.loc[
        (~test_predictions["correct"]) | test_predictions["prediction"].isna(),
        [
            "true_label_text",
            "prediction_text",
            "raw_output",
            "prompt_tokens",
            "seconds",
            "review_preview",
        ],
    ].head(20)
)


Confusion matrix:
[[11976   524]
 [  690 11810]]

              precision    recall  f1-score   support

    negative     0.9455    0.9581    0.9518     12500
    positive     0.9575    0.9448    0.9511     12500

    accuracy                         0.9514     25000
   macro avg     0.9515    0.9514    0.9514     25000
weighted avg     0.9515    0.9514    0.9514     25000

错误或无法解析的前 20 条：


,true_label_text,prediction_text,raw_output,prompt_tokens,seconds,review_preview
1,positive,negative,negative,270,0.051601,This is the latest entry in the long series of...
55,positive,negative,negative,191,0.043699,I remember seeing this film in the mid 80's th...
75,negative,positive,positive,108,0.038498,A truly masterful piece of filmmaking. It mana...
79,positive,negative,negative,152,0.038498,"First, the positives: an excellent job at depi..."
84,positive,negative,negative,211,0.043129,You can tell that this is the first offering b...
90,negative,positive,positive,392,0.043129,"Just watched this today on TCM, where the othe..."
98,positive,negative,negative,379,0.035967,There are few films that deal with things that...
103,positive,negative,negative,108,0.035967,Great cult flick for MST-3K types: Richard Boo...
104,positive,negative,negative,113,0.035967,I have fond memories of watching this when it ...
108,negative,positive,positive,95,0.035967,"This is, without a doubt, the most hilarious m..."


## 15. Save the Final Predictions and Metrics


In [11]:
test_predictions.to_csv(RESULTS_DIR / "lora_test_predictions.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR / "lora_test_metrics.csv", index=False)

with open(RESULTS_DIR / "lora_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("Adapter:", ADAPTER_DIR)
print("Results:", RESULTS_DIR)


Adapter: qwen3_imdb_lora\best_adapter
Results: qwen3_imdb_lora\results
